# Clean RoBERTa Retraining for BABE SG2

This replaces the confusing mixed-purpose `babe_dataset.ipynb` with one reproducible workflow.

**Goal:** train `roberta-base` on the cleaned BABE SG2 sentence-level binary task and evaluate once on an untouched article-disjoint test set.

- Labels: `0 = Non-biased`, `1 = Biased`
- Split: article-disjoint train / validation / test
- Validation is used for checkpoint selection and early stopping.
- Test is used only once for final reporting.
- Max token length: 256
- Save the selected model and exact split CSVs.

**Recommended runtime:** Google Colab with a GPU (Runtime → Change runtime type → T4 GPU, if available). The original notebook uses `/content/...` paths and assumes a Colab runtime. If you run locally on CPU, training will be much slower.

**Input:** `babe_cleaned.csv` produced by your cleaning workflow. It must contain `text`, `label_bias`, and `news_link`; `outlet` is recommended.


## 1. Install dependencies

In [ ]:
%pip install -q transformers datasets accelerate scikit-learn pandas numpy matplotlib

## 2. Load the cleaned dataset

Upload `babe_cleaned.csv` to Colab's current working directory, or update `DATA_PATH` to the correct file. This notebook does not silently download or merge SG1; use the cleaned SG2 dataset only.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_PATH = Path("/content/babe_cleaned.csv")
if not DATA_PATH.exists():
    # Also support running from the same folder as the CSV.
    DATA_PATH = Path("babe_cleaned.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Could not find babe_cleaned.csv. Upload it to Colab or change DATA_PATH."
    )

df = pd.read_csv(DATA_PATH)
print("Loaded:", DATA_PATH.resolve())
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head())

required = {"text", "label_bias", "news_link"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

## 3. Clean labels and duplicate text rows

In [ ]:
def normalize_label(value):
    if pd.isna(value):
        return np.nan
    if isinstance(value, (int, np.integer, float, np.floating)) and value in (0, 1):
        return int(value)
    label = str(value).strip().lower().replace("_", "-")
    mapping = {
        "non-biased": 0, "non biased": 0, "nonbiased": 0,
        "unbiased": 0, "0": 0,
        "biased": 1, "1": 1
    }
    return mapping.get(label, np.nan)

df["label"] = df["label_bias"].map(normalize_label)
df["text"] = df["text"].astype("string").str.strip()
df.loc[df["text"].eq(""), "text"] = pd.NA
df = df.dropna(subset=["text", "label", "news_link"]).copy()
df["label"] = df["label"].astype(int)
df["news_link"] = df["news_link"].astype(str).str.strip()
df = df[df["news_link"].ne("") & df["news_link"].ne("nan")].copy()

# Drop identical texts with conflicting labels rather than arbitrarily choosing one.
label_counts = df.groupby("text")["label"].nunique()
conflicting_texts = label_counts[label_counts > 1].index
print("Conflicting text groups excluded:", len(conflicting_texts))
df = df[~df["text"].isin(conflicting_texts)].copy()

# Exact duplicate text removed. Keep one row per sentence.
before = len(df)
df = df.drop_duplicates(subset=["text"], keep="first").reset_index(drop=True)
print("Duplicate text rows removed:", before - len(df))
print("Rows available for modeling:", len(df))
print("Label counts (0=non-biased, 1=biased):")
display(df["label"].value_counts().sort_index().to_frame("count"))
print("Label proportions:")
display(df["label"].value_counts(normalize=True).sort_index().round(3).to_frame("proportion"))

## 4. Create article-disjoint train / validation / test sets

All sentences sharing the same `news_link` stay in only one split. Missing links were removed above because they cannot reliably identify article groups. The two-stage split targets approximately 70% train, 15% validation, and 15% test; group sizes mean the exact percentages may vary.

Do not change or regenerate these splits after looking at final test performance. We save them so later experiments can reuse the same held-out rows.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

df["article_group"] = df["news_link"].astype(str)

# Reserve ~15% of article groups for the final test set.
split_test = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
trainval_idx, test_idx = next(
    split_test.split(df, y=df["label"], groups=df["article_group"])
)
trainval_df = df.iloc[trainval_idx].copy()
test_df = df.iloc[test_idx].copy()

# Reserve ~15% of total data for validation: 0.15 / 0.85 of trainval.
split_val = GroupShuffleSplit(n_splits=1, test_size=0.17647, random_state=43)
train_idx, val_idx = next(
    split_val.split(
        trainval_df,
        y=trainval_df["label"],
        groups=trainval_df["article_group"]
    )
)
train_df = trainval_df.iloc[train_idx].copy()
val_df = trainval_df.iloc[val_idx].copy()

train_groups = set(train_df["article_group"])
val_groups = set(val_df["article_group"])
test_groups = set(test_df["article_group"])

print("Rows — train:", len(train_df), "validation:", len(val_df), "test:", len(test_df))
print("Article overlap — train/val:", len(train_groups & val_groups))
print("Article overlap — train/test:", len(train_groups & test_groups))
print("Article overlap — val/test:", len(val_groups & test_groups))
assert not (train_groups & val_groups or train_groups & test_groups or val_groups & test_groups)

for name, part in [("Train", train_df), ("Validation", val_df), ("Test", test_df)]:
    print(f"\n{name}: {len(part)} rows; {part['article_group'].nunique()} articles")
    display(part["label"].value_counts(normalize=True).sort_index().round(3).rename(index={0:"non-biased",1:"biased"}).to_frame("proportion"))

# Save the exact splits so they can be reused for baseline models and evaluation.
SPLIT_DIR = Path("/content/babe_sg2_splits")
if not Path("/content").exists():
    SPLIT_DIR = Path("babe_sg2_splits")
SPLIT_DIR.mkdir(parents=True, exist_ok=True)
for name, part in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    part.drop(columns=["article_group"], errors="ignore").to_csv(SPLIT_DIR / f"{name}.csv", index=False)
print("Split CSVs saved in:", SPLIT_DIR.resolve())

## 5. Check hardware

In [ ]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: CPU training is possible but will be slow. For a student project, use Colab GPU if available.")

## 6. Tokenize the three splits

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

MODEL_NAME = "FacebookAI/roberta-base"
MAX_LENGTH = 256
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def make_hf_dataset(frame):
    small = frame[["text", "label"]].copy()
    small["text"] = small["text"].fillna("").astype(str)
    small["label"] = small["label"].astype(int)
    return Dataset.from_pandas(small.reset_index(drop=True), preserve_index=False)

train_hf = make_hf_dataset(train_df)
val_hf = make_hf_dataset(val_df)
test_hf = make_hf_dataset(test_df)

def tokenize_batch(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)

train_hf = train_hf.map(tokenize_batch, batched=True, remove_columns=["text"])
val_hf = val_hf.map(tokenize_batch, batched=True, remove_columns=["text"])
test_hf = test_hf.map(tokenize_batch, batched=True, remove_columns=["text"])

collator = DataCollatorWithPadding(tokenizer=tokenizer)
print("Tokenized rows:", len(train_hf), len(val_hf), len(test_hf))

## 7. Train RoBERTa using validation only for checkpoint selection

This is the important fix compared with the first workflow in `babe_dataset.ipynb`: `eval_dataset=val_hf`, not `test_hf`. The final test set is not passed into the Trainer during training.

In [ ]:
import os
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

MODEL_OUTPUT_DIR = "/content/newsbias_roberta_retrain"
if not Path("/content").exists():
    MODEL_OUTPUT_DIR = "newsbias_roberta_retrain"

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: "Non-biased", 1: "Biased"},
    label2id={"Non-biased": 0, "Biased": 1}
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "macro_f1": f1_score(labels, predictions, average="macro", zero_division=0)
    }

use_fp16 = torch.cuda.is_available()
args = TrainingArguments(
    output_dir=MODEL_OUTPUT_DIR,
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_macro_f1",
    greater_is_better=True,
    fp16=use_fp16,
    logging_steps=50,
    report_to="none",
    seed=42,
    data_seed=42,
    save_total_limit=2
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_hf,
    eval_dataset=val_hf,
    data_collator=collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)]
)

train_result = trainer.train()
print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Best validation macro-F1:", trainer.state.best_metric)

## 8. Final evaluation — run once on the untouched test set

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

test_output = trainer.predict(test_hf)
y_true = test_output.label_ids
y_pred = np.argmax(test_output.predictions, axis=-1)

final_accuracy = accuracy_score(y_true, y_pred)
final_macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
print("FINAL TEST ACCURACY:", round(final_accuracy, 4))
print("FINAL TEST MACRO-F1:", round(final_macro_f1, 4))
print("\nClassification report:")
print(classification_report(
    y_true, y_pred, labels=[0, 1],
    target_names=["Non-biased", "Biased"], digits=4, zero_division=0
))

ConfusionMatrixDisplay.from_predictions(
    y_true, y_pred, labels=[0, 1],
    display_labels=["Non-biased", "Biased"], cmap="Blues"
)
plt.title("RoBERTa — Article-disjoint final test")
plt.tight_layout()
plt.show()

## 9. Save the final model and test predictions

In [ ]:
FINAL_MODEL_DIR = "/content/newsbias_roberta_best_retrained"
if not Path("/content").exists():
    FINAL_MODEL_DIR = "newsbias_roberta_best_retrained"

trainer.save_model(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)
print("Saved model:", Path(FINAL_MODEL_DIR).resolve())

test_predictions = test_df[["text", "label", "news_link"]].reset_index(drop=True).copy()
test_predictions["predicted_label"] = y_pred
test_predictions["correct"] = test_predictions["label"] == test_predictions["predicted_label"]
test_predictions["true_label_name"] = test_predictions["label"].map({0:"Non-biased", 1:"Biased"})
test_predictions["predicted_label_name"] = test_predictions["predicted_label"].map({0:"Non-biased", 1:"Biased"})
pred_path = Path(FINAL_MODEL_DIR) / "test_predictions.csv"
test_predictions.to_csv(pred_path, index=False)
print("Saved test predictions:", pred_path.resolve())
display(test_predictions[~test_predictions["correct"]].head(20))

## 10. What to report

Report the **final test** accuracy, macro-F1, per-class precision/recall/F1, and confusion matrix. Also report the split method (article-disjoint), dataset size after cleaning, maximum token length, learning rate, batch size, epoch/checkpoint selection rule, and random seeds.

Do not use the final test results to select hyperparameters. If you change the model or training settings after looking at test results, treat that test set as development data and create a new untouched test set for a final claim.

**Fair baseline comparison:** rerun Logistic Regression and SVM using the saved `train.csv` and `test.csv` in `babe_sg2_splits`, not the separate random split. The baseline and RoBERTa should use the same test sentences.
